In [5]:
import pandas as pd

In [6]:
df = pd.read_csv("../data/raw/council_voting_records.csv", sep=";")

df.head()

,meeting_id,meeting_type,vote_date,vote_number,agenda_description,vote_start_date_time,council_member,vote,decision,vote_detail_id
0,18660,Council,2025-06-17,10770,Local Improvement â€“ Project List of Consider...,2025-06-17T23:08:18+00:00,Councillor R Bligh,In Favour,Carried Unanimously,104725
1,18660,Council,2025-06-17,10770,Local Improvement â€“ Project List of Consider...,2025-06-17T23:08:18+00:00,Councillor P Meiszner,In Favour,Carried Unanimously,104729
2,18660,Council,2025-06-17,10770,Local Improvement â€“ Project List of Consider...,2025-06-17T23:08:18+00:00,Councillor L Zhou,In Favour,Carried Unanimously,104730
3,18660,Council,2025-06-17,10771,Selection of and Terms for Social Housing Oper...,2025-06-17T23:12:57+00:00,Councillor L Dominato,In Favour,Carried Unanimously,104735
4,18660,Council,2025-06-17,10771,Selection of and Terms for Social Housing Oper...,2025-06-17T23:12:57+00:00,Councillor L Zhou,In Favour,Carried Unanimously,104741


In [7]:
df.shape

(83824, 10)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 83824 entries, 0 to 83823
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   meeting_id            83824 non-null  int64
 1   meeting_type          83824 non-null  str  
 2   vote_date             83824 non-null  str  
 3   vote_number           83824 non-null  int64
 4   agenda_description    83824 non-null  str  
 5   vote_start_date_time  83824 non-null  str  
 6   council_member        83824 non-null  str  
 7   vote                  83824 non-null  str  
 8   decision              83824 non-null  str  
 9   vote_detail_id        83824 non-null  int64
dtypes: int64(3), str(7)
memory usage: 6.4 MB


In [9]:
df.describe(include="all")

,meeting_id,meeting_type,vote_date,vote_number,agenda_description,vote_start_date_time,council_member,vote,decision,vote_detail_id
count,83824.000000,83824,83824,83824.000000,83824,83824,83824,83824,83824,83824.000000
unique,NaN,8,622,NaN,6524,7660,29,7,3,NaN
top,NaN,Council,2018-12-18,NaN,1. Mayor's Overdose Emergency Task Force,2023-11-01T17:22:09+00:00,Councillor A Carr,In Favour,Carried Unanimously,NaN
freq,NaN,24324,693,NaN,330,22,6475,66824,59332,NaN
mean,18071.822533,NaN,NaN,6684.363953,NaN,NaN,NaN,NaN,NaN,62046.713125
std,405.552033,NaN,NaN,3162.116171,NaN,NaN,NaN,NaN,NaN,34980.524063
min,17431.000000,NaN,NaN,1121.000000,NaN,NaN,NaN,NaN,NaN,1552.000000
25%,17748.000000,NaN,NaN,3960.000000,NaN,NaN,NaN,NaN,NaN,32598.750000
50%,18004.000000,NaN,NaN,6824.000000,NaN,NaN,NaN,NaN,NaN,62409.500000
75%,18404.000000,NaN,NaN,9488.000000,NaN,NaN,NaN,NaN,NaN,90800.250000


In [11]:
categorical_cols = [
    "meeting_type",
    "council_member",
    "vote",
    "decision"
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- meeting_type ---
meeting_type
Council                          24324
Policy & Strategic Priorities    17035
City Finance & Services          15156
Regular Council                  14115
Public Hearing                   10095
Special Council                   2673
Courts of Revision                 220
Auditor General Committee          206
Name: count, dtype: int64

--- council_member ---
council_member
Councillor A Carr          6475
Councillor P Fry           6116
Councillor S Kirby-Yung    6116
Councillor L Dominato      6112
Councillor R Bligh         6087
Councillor M De Genova     5215
Councillor C Boyle         4968
Councillor M Wiebe         3670
Councillor C Hardwick      3670
Mayor K Stewart            3666
Councillor J Swanson       3666
Councillor B Montague      2455
Councillor L Zhou          2431
Councillor M Klassen       2426
Councillor P Meiszner      2416
Mayor K Sim                2416
Mayor G Robertson          1549
Councillor A Reimer        1549
Councillor E

In [12]:
# Select all string columns
text_cols = df.select_dtypes(include=["object", "str"]).columns

for col in text_cols:
    # Count whitespace-only values
    blank_count = df[col].str.strip().eq("").sum()

    # Count values with leading/trailing whitespace
    whitespace_count = (
        df[col].ne(df[col].str.strip())
    ).sum()

    print(f"{col}:")
    print(f"  Blank strings: {blank_count}")
    print(f"  Leading/trailing whitespace: {whitespace_count}")

meeting_type:
  Blank strings: 0
  Leading/trailing whitespace: 0
vote_date:
  Blank strings: 0
  Leading/trailing whitespace: 0
agenda_description:
  Blank strings: 0
  Leading/trailing whitespace: 0
vote_start_date_time:
  Blank strings: 0
  Leading/trailing whitespace: 0
council_member:
  Blank strings: 0
  Leading/trailing whitespace: 0
vote:
  Blank strings: 0
  Leading/trailing whitespace: 0
decision:
  Blank strings: 0
  Leading/trailing whitespace: 0


In [13]:
# Check for completely duplicated rows
print("Exact duplicate rows:", df.duplicated().sum())

# Check for duplicated vote detail IDs
print(
    "Duplicate vote_detail_id:",
    df["vote_detail_id"].duplicated().sum()
)

# Check the uniqueness of vote_detail_id
print(
    "Unique vote_detail_id:",
    df["vote_detail_id"].nunique()
)

Exact duplicate rows: 0
Duplicate vote_detail_id: 0
Unique vote_detail_id: 83824


In [14]:
# Convert date column
vote_dates = pd.to_datetime(
    df["vote_date"],
    errors="coerce"
)

# Convert timestamp column
vote_timestamps = pd.to_datetime(
    df["vote_start_date_time"],
    errors="coerce",
    utc=True
)

# Check conversion results
print("Invalid vote dates:", vote_dates.isna().sum())
print("Invalid vote timestamps:", vote_timestamps.isna().sum())

print("\nDate range:")
print("Earliest:", vote_dates.min())
print("Latest:", vote_dates.max())

Invalid vote dates: 0
Invalid vote timestamps: 0

Date range:
Earliest: 2016-04-05 00:00:00
Latest: 2026-10-27 00:00:00


In [15]:
df["vote_date"] = vote_dates
df["vote_start_date_time"] = vote_timestamps

df[["vote_date", "vote_start_date_time"]].dtypes

vote_date                    datetime64[us]
vote_start_date_time    datetime64[us, UTC]
dtype: object

In [16]:
local_dates = (
    df["vote_start_date_time"]
    .dt.tz_convert("America/Vancouver")
    .dt.date
)

date_mismatches = (
    local_dates != df["vote_date"].dt.date
)

print("Date mismatches:", date_mismatches.sum())

df.loc[
    date_mismatches,
    ["vote_date", "vote_start_date_time"]
].head(10)

Date mismatches: 11


,vote_date,vote_start_date_time
4736,2026-02-17,2026-02-18 08:02:49+00:00
4737,2026-02-17,2026-02-18 08:02:49+00:00
4738,2026-02-17,2026-02-18 08:02:49+00:00
4739,2026-02-17,2026-02-18 08:02:49+00:00
4740,2026-02-17,2026-02-18 08:02:49+00:00
4741,2026-02-17,2026-02-18 08:02:49+00:00
4742,2026-02-17,2026-02-18 08:02:49+00:00
32687,2026-02-17,2026-02-18 08:02:49+00:00
32688,2026-02-17,2026-02-18 08:02:49+00:00
73201,2026-02-17,2026-02-18 08:02:49+00:00


In [17]:
df.loc[
    date_mismatches,
    [
        "meeting_id",
        "meeting_type",
        "vote_date",
        "vote_number",
        "agenda_description",
        "vote_start_date_time",
        "council_member",
        "vote",
        "decision"
    ]
].sort_values("vote_number")

,meeting_id,meeting_type,vote_date,vote_number,agenda_description,vote_start_date_time,council_member,vote,decision
4736,18872,Public Hearing,2026-02-17,11400,CD-1 (899) Text Amendment: 3360-3384 Vanness A...,2026-02-18 08:02:49+00:00,Mayor K Sim,Absent,Carried Unanimously
4737,18872,Public Hearing,2026-02-17,11400,CD-1 (899) Text Amendment: 3360-3384 Vanness A...,2026-02-18 08:02:49+00:00,Councillor S Kirby-Yung,In Favour,Carried Unanimously
4738,18872,Public Hearing,2026-02-17,11400,CD-1 (899) Text Amendment: 3360-3384 Vanness A...,2026-02-18 08:02:49+00:00,Councillor L Dominato,In Favour,Carried Unanimously
4739,18872,Public Hearing,2026-02-17,11400,CD-1 (899) Text Amendment: 3360-3384 Vanness A...,2026-02-18 08:02:49+00:00,Councillor R Bligh,Absent,Carried Unanimously
4740,18872,Public Hearing,2026-02-17,11400,CD-1 (899) Text Amendment: 3360-3384 Vanness A...,2026-02-18 08:02:49+00:00,Councillor P Fry,In Favour,Carried Unanimously
4741,18872,Public Hearing,2026-02-17,11400,CD-1 (899) Text Amendment: 3360-3384 Vanness A...,2026-02-18 08:02:49+00:00,Councillor P Meiszner,In Favour,Carried Unanimously
4742,18872,Public Hearing,2026-02-17,11400,CD-1 (899) Text Amendment: 3360-3384 Vanness A...,2026-02-18 08:02:49+00:00,Councillor S Orr,Abstain,Carried Unanimously
32687,18872,Public Hearing,2026-02-17,11400,CD-1 (899) Text Amendment: 3360-3384 Vanness A...,2026-02-18 08:02:49+00:00,Councillor L Zhou,In Favour,Carried Unanimously
32688,18872,Public Hearing,2026-02-17,11400,CD-1 (899) Text Amendment: 3360-3384 Vanness A...,2026-02-18 08:02:49+00:00,Councillor L Maloney,In Favour,Carried Unanimously
73201,18872,Public Hearing,2026-02-17,11400,CD-1 (899) Text Amendment: 3360-3384 Vanness A...,2026-02-18 08:02:49+00:00,Councillor B Montague,In Favour,Carried Unanimously


In [18]:
print(
    "Unique meetings:",
    df.loc[date_mismatches, "meeting_id"].nunique()
)

print(
    "Unique vote numbers:",
    df.loc[date_mismatches, "vote_number"].nunique()
)

Unique meetings: 1
Unique vote numbers: 1


In [19]:
# Number of unique identifiers
print("Unique meetings:", df["meeting_id"].nunique())
print("Unique vote numbers:", df["vote_number"].nunique())
print("Unique vote detail IDs:", df["vote_detail_id"].nunique())

# Check whether any vote number belongs to multiple meetings
vote_meeting_counts = (
    df.groupby("vote_number")["meeting_id"]
    .nunique()
)

print(
    "\nVote numbers associated with multiple meetings:",
    (vote_meeting_counts > 1).sum()
)

print(
    "\nMaximum meetings per vote number:",
    vote_meeting_counts.max()
)

Unique meetings: 745
Unique vote numbers: 7665
Unique vote detail IDs: 83824

Vote numbers associated with multiple meetings: 0

Maximum meetings per vote number: 1


In [20]:
vote_level_cols = [
    "meeting_id",
    "meeting_type",
    "vote_date",
    "agenda_description",
    "vote_start_date_time",
    "decision"
]

for col in vote_level_cols:
    inconsistent_votes = (
        df.groupby("vote_number")[col]
        .nunique(dropna=False)
        .gt(1)
        .sum()
    )

    print(f"{col}: {inconsistent_votes} inconsistent votes")

meeting_id: 0 inconsistent votes
meeting_type: 0 inconsistent votes
vote_date: 0 inconsistent votes
agenda_description: 0 inconsistent votes
vote_start_date_time: 0 inconsistent votes
decision: 0 inconsistent votes


In [21]:
# Check duplicate councillor-vote combinations
duplicate_member_votes = df.duplicated(
    subset=["vote_number", "council_member"],
    keep=False
)

print(
    "Records with duplicate councillor-vote combinations:",
    duplicate_member_votes.sum()
)

print(
    "Affected vote numbers:",
    df.loc[duplicate_member_votes, "vote_number"].nunique()
)

# Display any problematic records
df.loc[
    duplicate_member_votes,
    ["vote_number", "council_member", "vote", "vote_detail_id"]
].sort_values(["vote_number", "council_member"]).head(20)

Records with duplicate councillor-vote combinations: 42
Affected vote numbers: 7


,vote_number,council_member,vote,vote_detail_id
3255,8679,Councillor A Carr,No Vote,82072
49063,8679,Councillor A Carr,No Vote,82074
78068,8679,Councillor A Carr,No Vote,82071
78069,8679,Councillor A Carr,No Vote,82073
3257,8680,Councillor A Carr,No Vote,82084
49066,8680,Councillor A Carr,No Vote,82083
49067,8680,Councillor A Carr,No Vote,82085
78072,8680,Councillor A Carr,No Vote,82082
7141,9604,Councillor A Carr,In Favour,92074
75279,9604,Councillor A Carr,In Favour,92250


In [23]:
# Extract all records with repeated councillor-vote combinations
duplicates = df.loc[duplicate_member_votes].copy()

# Check whether duplicated combinations have different voting statuses
vote_disagreements = (
    duplicates.groupby(["vote_number", "council_member"])["vote"]
    .nunique()
)

print(
    "Combinations with conflicting voting statuses:",
    (vote_disagreements > 1).sum()
)

# Check whether records are identical except for vote_detail_id
columns_to_compare = [
    col for col in df.columns if col != "vote_detail_id"
]

print(
    "Duplicate records ignoring vote_detail_id:",
    duplicates.duplicated(
        subset=columns_to_compare
    ).sum()
)

# Summarize affected votes
print("\nAffected vote numbers:")
print(
    duplicates.groupby("vote_number").agg(
        records=("vote_detail_id", "size"),
        councillors=("council_member", "nunique"),
        voting_statuses=("vote", "nunique")
    )
)

Combinations with conflicting voting statuses: 3
Duplicate records ignoring vote_detail_id: 22

Affected vote numbers:
             records  councillors  voting_statuses
vote_number                                       
8679               4            1                1
8680               4            1                1
9604              22           11                3
10602              3            1                2
10604              3            1                2
10703              3            1                1
11024              3            1                2


In [24]:
# Identify combinations with conflicting voting statuses
conflicting_combinations = vote_disagreements[
    vote_disagreements > 1
].index

# Display all records associated with those combinations
conflicting_records = (
    df.set_index(["vote_number", "council_member"])
    .loc[conflicting_combinations]
    .reset_index()
)

conflicting_records[
    [
        "meeting_id",
        "vote_date",
        "vote_number",
        "agenda_description",
        "council_member",
        "vote",
        "vote_detail_id"
    ]
].sort_values(["vote_number", "vote_detail_id"])

,meeting_id,vote_date,vote_number,agenda_description,council_member,vote,vote_detail_id
2,18642,2025-04-01,10602,R2. 2025 Animal Welfare Grant,Councillor B Montague,No Vote,102908
1,18642,2025-04-01,10602,R2. 2025 Animal Welfare Grant,Councillor B Montague,No Vote,102912
0,18642,2025-04-01,10602,R2. 2025 Animal Welfare Grant,Councillor B Montague,In Favour,102914
5,18642,2025-04-01,10604,C1. Change to Council Meeting Schedule,Councillor B Montague,No Vote,102930
4,18642,2025-04-01,10604,C1. Change to Council Meeting Schedule,Councillor B Montague,No Vote,102934
3,18642,2025-04-01,10604,C1. Change to Council Meeting Schedule,Councillor B Montague,In Favour,102936
8,18724,2025-10-09,11024,Office of the Auditor General 2026 Budget Request,Councillor B Montague,No Vote,116137
6,18724,2025-10-09,11024,Office of the Auditor General 2026 Budget Request,Councillor B Montague,No Vote,116138
7,18724,2025-10-09,11024,Office of the Auditor General 2026 Budget Request,Councillor B Montague,In Favour,116140


In [25]:
# Examine the full voting records for the three affected votes
affected_votes = [10602, 10604, 11024]

for vote_num in affected_votes:
    print(f"\n--- Vote {vote_num} ---")

    vote_records = df.loc[
        df["vote_number"] == vote_num,
        ["council_member", "vote", "vote_detail_id", "decision"]
    ]

    print(
        vote_records
        .sort_values(["council_member", "vote_detail_id"])
        .to_string(index=False)
    )


--- Vote 10602 ---
         council_member      vote  vote_detail_id            decision
  Councillor B Montague   No Vote          102908 Carried Unanimously
  Councillor B Montague   No Vote          102912 Carried Unanimously
  Councillor B Montague In Favour          102914 Carried Unanimously
  Councillor L Dominato    Absent          102910 Carried Unanimously
      Councillor L Zhou In Favour          102917 Carried Unanimously
   Councillor M Klassen In Favour          102915 Carried Unanimously
       Councillor P Fry    Absent          102913 Carried Unanimously
  Councillor P Meiszner In Favour          102916 Carried Unanimously
     Councillor R Bligh In Favour          102911 Carried Unanimously
Councillor S Kirby-Yung In Favour          102909 Carried Unanimously
            Mayor K Sim In Favour          102907 Carried Unanimously

--- Vote 10604 ---
         council_member      vote  vote_detail_id            decision
  Councillor B Montague   No Vote          102930 

In [26]:
remaining_votes = [8679, 8680, 9604, 10703]

for vote_num in remaining_votes:
    print(f"\n--- Vote {vote_num} ---")

    records = df.loc[
        df["vote_number"] == vote_num
    ]

    print("Total records:", len(records))
    print("Unique councillors:", records["council_member"].nunique())

    print(
        records[
            ["council_member", "vote", "vote_detail_id"]
        ]
        .sort_values(["council_member", "vote_detail_id"])
        .to_string(index=False)
    )


--- Vote 8679 ---
Total records: 9
Unique councillors: 6
         council_member      vote  vote_detail_id
      Councillor A Carr   No Vote           82071
      Councillor A Carr   No Vote           82072
      Councillor A Carr   No Vote           82073
      Councillor A Carr   No Vote           82074
  Councillor C Hardwick In Favour           82075
     Councillor M Wiebe    Absent           82076
       Councillor P Fry In Favour           82070
     Councillor R Bligh In Favour           82079
Councillor S Kirby-Yung In Favour           82080

--- Vote 8680 ---
Total records: 9
Unique councillors: 6
         council_member      vote  vote_detail_id
      Councillor A Carr   No Vote           82082
      Councillor A Carr   No Vote           82083
      Councillor A Carr   No Vote           82084
      Councillor A Carr   No Vote           82085
  Councillor C Hardwick In Favour           82086
     Councillor M Wiebe    Absent           82087
       Councillor P Fry In Favour 

In [27]:
# Number of extra records beyond one per councillor per vote
extra_records = df.duplicated(
    subset=["vote_number", "council_member"]
).sum()

print("Total extra records:", extra_records)

# Count extras for each affected vote
extras_by_vote = (
    df.groupby(["vote_number", "council_member"])
    .size()
    .sub(1)
    .clip(lower=0)
    .groupby(level="vote_number")
    .sum()
)

print("\nExtra records by vote:")
print(extras_by_vote[extras_by_vote > 0])

print(
    "\nRows after hypothetical deduplication:",
    len(df) - extra_records
)

Total extra records: 25

Extra records by vote:
vote_number
8679      3
8680      3
9604     11
10602     2
10604     2
10703     2
11024     2
dtype: int64

Rows after hypothetical deduplication: 83799


In [28]:
# Check meeting-level consistency
meeting_cols = ["meeting_type", "vote_date"]

for col in meeting_cols:
    counts = (
        df.groupby("meeting_id")[col]
        .nunique(dropna=False)
    )

    print(
        f"{col}: {(counts > 1).sum()} meetings with multiple values"
    )

# Inspect meetings associated with multiple vote dates
meeting_dates = df.groupby("meeting_id")["vote_date"].nunique()

print("\nMeetings with multiple vote dates:")
print(meeting_dates[meeting_dates > 1].head(20))

meeting_type: 0 meetings with multiple values
vote_date: 2 meetings with multiple values

Meetings with multiple vote dates:
meeting_id
17704    2
17828    2
Name: vote_date, dtype: int64


In [29]:
meetings_to_check = [17704, 17828]

meeting_summary = (
    df[df["meeting_id"].isin(meetings_to_check)]
    .groupby(["meeting_id", "meeting_type", "vote_date"])
    .agg(
        unique_votes=("vote_number", "nunique"),
        records=("vote_detail_id", "count"),
        earliest_timestamp=("vote_start_date_time", "min"),
        latest_timestamp=("vote_start_date_time", "max")
    )
    .reset_index()
)

meeting_summary

,meeting_id,meeting_type,vote_date,unique_votes,records,earliest_timestamp,latest_timestamp
0,17704,Policy & Strategic Priorities,2018-04-18,5,55,2018-04-19 03:47:33+00:00,2018-04-19 03:48:41+00:00
1,17704,Policy & Strategic Priorities,2018-04-19,2,22,2018-04-20 03:43:52+00:00,2018-04-20 03:45:48+00:00
2,17828,City Finance & Services,2019-04-24,1,11,2019-04-25 04:32:26+00:00,2019-04-25 04:32:26+00:00
3,17828,City Finance & Services,2019-04-29,13,143,2019-04-30 02:23:21+00:00,2019-04-30 05:33:51+00:00


In [30]:
# Count individual voting records by status and decision
vote_decision_counts = pd.crosstab(
    df["vote"],
    df["decision"]
)

vote_decision_counts

decision,Carried,Carried Unanimously,Lost
vote,,,
Absent,1294,6080,449
Abstain,410,479,114
Declared Conflict,86,196,25
In Favour,12882,52133,1809
In Opposition,3617,27,3746
Ineligible,5,15,3
No Vote,47,402,5


In [31]:
# Identify records marked In Opposition
# but associated with a unanimous decision
unusual_votes = df[
    (df["vote"] == "In Opposition") &
    (df["decision"] == "Carried Unanimously")
]

print("Total unusual records:", len(unusual_votes))

print(
    "Distinct affected votes:",
    unusual_votes["vote_number"].nunique()
)

print(
    "Distinct affected meetings:",
    unusual_votes["meeting_id"].nunique()
)

# Examine the affected vote numbers and councillors
unusual_votes[
    [
        "meeting_id",
        "vote_date",
        "vote_number",
        "agenda_description",
        "council_member",
        "vote",
        "decision"
    ]
].sort_values("vote_number")

Total unusual records: 27
Distinct affected votes: 10
Distinct affected meetings: 10


,meeting_id,vote_date,vote_number,agenda_description,council_member,vote,decision
29659,17449,2016-06-14,1292,Report Reference 1 - Playland Redevelopment Pl...,Councillor T Stevenson,In Opposition,Carried Unanimously
59929,17449,2016-06-14,1292,Report Reference 1 - Playland Redevelopment Pl...,Councillor R Louie,In Opposition,Carried Unanimously
29662,17449,2016-06-14,1292,Report Reference 1 - Playland Redevelopment Pl...,Councillor G Meggs,In Opposition,Carried Unanimously
59930,17449,2016-06-14,1292,Report Reference 1 - Playland Redevelopment Pl...,Councillor K Jang,In Opposition,Carried Unanimously
29658,17449,2016-06-14,1292,Report Reference 1 - Playland Redevelopment Pl...,Mayor G Robertson,In Opposition,Carried Unanimously
58669,17523,2017-02-08,1750,6. Opioid Overdose Crisis Update - Postpone,Councillor M De Genova,In Opposition,Carried Unanimously
58666,17523,2017-02-08,1750,6. Opioid Overdose Crisis Update - Postpone,Councillor E Ball,In Opposition,Carried Unanimously
58668,17523,2017-02-08,1750,6. Opioid Overdose Crisis Update - Postpone,Councillor G Affleck,In Opposition,Carried Unanimously
36770,17526,2017-02-22,1796,4. Mayor's Task Force on Mental Health & Addic...,Councillor M De Genova,In Opposition,Carried Unanimously
58616,17526,2017-02-22,1796,4. Mayor's Task Force on Mental Health & Addic...,Councillor G Affleck,In Opposition,Carried Unanimously


In [32]:
affected_vote_numbers = unusual_votes["vote_number"].unique()

affected_records = df[
    df["vote_number"].isin(affected_vote_numbers)
]

vote_summary = pd.crosstab(
    affected_records["vote_number"],
    affected_records["vote"]
)

vote_summary["decision"] = (
    affected_records.groupby("vote_number")["decision"].first()
)

vote_summary

vote,Absent,Declared Conflict,In Favour,In Opposition,decision
vote_number,,,,,
1292,2,0,4,5,Carried Unanimously
1750,1,0,7,3,Carried Unanimously
1796,1,0,8,2,Carried Unanimously
1923,0,0,10,1,Carried Unanimously
1991,1,0,7,3,Carried Unanimously
2434,0,1,8,2,Carried Unanimously
2881,1,0,9,1,Carried Unanimously
3293,3,0,6,2,Carried Unanimously
8807,0,0,9,2,Carried Unanimously


In [33]:
# Summarize the number of votes in favour and opposition
vote_counts = (
    df.groupby(["vote_number", "decision", "vote"])
    .size()
    .unstack(fill_value=0)
)

# Identify votes marked Lost with:
# - At least one In Favour vote
# - Zero In Opposition votes

unusual_lost_votes = vote_counts[
    (vote_counts.index.get_level_values("decision") == "Lost")
    & (vote_counts["In Favour"] > 0)
    & (vote_counts["In Opposition"] == 0)
]

print("Potentially inconsistent Lost votes:", len(unusual_lost_votes))

unusual_lost_votes

Potentially inconsistent Lost votes: 1


,vote,Absent,Abstain,Declared Conflict,In Favour,In Opposition,Ineligible,No Vote
vote_number,decision,,,,,,,
9848,Lost,1,0,0,10,0,0,0


In [34]:
df.loc[
    df["vote_number"] == 9848,
    [
        "meeting_id",
        "meeting_type",
        "vote_date",
        "vote_number",
        "agenda_description",
        "council_member",
        "vote",
        "decision"
    ]
].sort_values("council_member")

,meeting_id,meeting_type,vote_date,vote_number,agenda_description,council_member,vote,decision
6375,18493,Council,2024-03-12,9848,CD-1 Rezoning: 255-285 Southwest Marine Drive,Councillor A Carr,In Favour,Lost
50097,18493,Council,2024-03-12,9848,CD-1 Rezoning: 255-285 Southwest Marine Drive,Councillor B Montague,In Favour,Lost
6377,18493,Council,2024-03-12,9848,CD-1 Rezoning: 255-285 Southwest Marine Drive,Councillor C Boyle,In Favour,Lost
50095,18493,Council,2024-03-12,9848,CD-1 Rezoning: 255-285 Southwest Marine Drive,Councillor L Dominato,In Favour,Lost
50100,18493,Council,2024-03-12,9848,CD-1 Rezoning: 255-285 Southwest Marine Drive,Councillor L Zhou,In Favour,Lost
50098,18493,Council,2024-03-12,9848,CD-1 Rezoning: 255-285 Southwest Marine Drive,Councillor M Klassen,In Favour,Lost
6378,18493,Council,2024-03-12,9848,CD-1 Rezoning: 255-285 Southwest Marine Drive,Councillor P Fry,In Favour,Lost
50099,18493,Council,2024-03-12,9848,CD-1 Rezoning: 255-285 Southwest Marine Drive,Councillor P Meiszner,In Favour,Lost
50096,18493,Council,2024-03-12,9848,CD-1 Rezoning: 255-285 Southwest Marine Drive,Councillor R Bligh,In Favour,Lost
6376,18493,Council,2024-03-12,9848,CD-1 Rezoning: 255-285 Southwest Marine Drive,Councillor S Kirby-Yung,In Favour,Lost


In [35]:
records_per_vote = (
    df.groupby("vote_number")
    .agg(
        total_records=("vote_detail_id", "count"),
        unique_members=("council_member", "nunique")
    )
)

print(records_per_vote.describe())

print("\nDistribution of unique members per vote:")
print(
    records_per_vote["unique_members"]
    .value_counts()
    .sort_index()
)

print("\nVotes with fewer than 5 unique members:")
print(
    records_per_vote[
        records_per_vote["unique_members"] < 5
    ]
)

       total_records  unique_members
count    7665.000000     7665.000000
mean       10.935943       10.932681
std         0.468411        0.460788
min         4.000000        4.000000
25%        11.000000       11.000000
50%        11.000000       11.000000
75%        11.000000       11.000000
max        22.000000       11.000000

Distribution of unique members per vote:
unique_members
4        1
5       18
6       15
7        1
9      126
10      70
11    7434
Name: count, dtype: int64

Votes with fewer than 5 unique members:
             total_records  unique_members
vote_number                               
11541                    4               4


In [36]:
df.loc[
    df["vote_number"] == 11541,
    [
        "meeting_id",
        "meeting_type",
        "vote_date",
        "vote_number",
        "agenda_description",
        "council_member",
        "vote",
        "decision"
    ]
]

,meeting_id,meeting_type,vote_date,vote_number,agenda_description,council_member,vote,decision
4390,18838,Special Council,2026-04-21,11541,1. Business Licence Application No. AL0188353 ...,Councillor B Montague,In Favour,Carried Unanimously
32333,18838,Special Council,2026-04-21,11541,1. Business Licence Application No. AL0188353 ...,Councillor R Bligh,Absent,Carried Unanimously
72852,18838,Special Council,2026-04-21,11541,1. Business Licence Application No. AL0188353 ...,Councillor L Zhou,In Favour,Carried Unanimously
72853,18838,Special Council,2026-04-21,11541,1. Business Licence Application No. AL0188353 ...,Councillor S Orr,In Favour,Carried Unanimously
